# M2-B2 — Biais du modèle d'annulation (binôme)

Le calcul est fourni ; les constats, les choix et leur interprétation sont à rédiger dans les cellules ✏️.

In [5]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib
from features import INPUT_FEATURES
from modele import entrainer, mesures_par_groupe

train = pd.read_csv(ROOT / "data/reservations_train.csv")
test = pd.read_csv(ROOT / "data/reservations_test.csv")
v21 = joblib.load(ROOT / "model/cancellation_model_v2.1.0.joblib")
len(train), len(test)

(58199, 19694)

## Tâche 2 — Mesurer

Groupes : clients portugais (`PRT`) et autres pays (pays manquant compris).

In [6]:
groupe = test["country"].eq("PRT").map({True: "Portugal", False: "Autres pays"})
mesures_par_groupe(v21, test, INPUT_FEATURES, groupe)

,effectif,annulations_reelles,taux_alertes,rappel,fausses_alertes,precision
Autres pays,12871.0,0.258,0.223,0.530,0.117,0.612
Portugal,6823.0,0.620,0.744,0.931,0.438,0.776


### ✏️ Constats

Que dit chaque colonne, en une phrase compréhensible par le service réservations ? Parmi les clients qui **viendront**, qui reçoit le plus souvent une alerte à tort ?

- *annulations_reelles* : les clients portugais annulent 62 % de leurs réservations, contre 26 % pour les autres pays.
- *taux_alertes* : le modèle signale 74 % des réservations portugaises, contre 22 % des autres. 
- *rappel* : le modèle repère 93 % des annulations portugaises, mais seulement 53 % des annulations des autres pays.
- *fausses_alertes* : parmi les clients qui viennent, 44 % des Portugais sont signalés à tort, contre 12 % des autres.
- *precision* : sur 100 réservations portugaises signalées, 77 sont réellement annulées. Pour les autres pays, 61 sur 100.

## Tâche 3 — Débattre

| | Arguments |
|---|---|
| Position A : l'écart est acceptable | Les taux réels diffèrent : 62 % d'annulations pour le Portugal, 26 % pour les autres pays.<br>Une alerte sur un client portugais est plus souvent juste (précision 78 % contre 61 %).<br>Le modèle repère 93 % des annulations portugaises : le service en rate peu. |
| Position B : l'écart est injuste | Parmi les clients qui viennent, un Portugais est signalé à tort plus de 3 fois plus souvent (44 % contre 12 %).<br>Le modèle accentue l'écart réel : 74 % d'alertes pour 62 % d'annulations côté Portugal, 22 % pour 26 % côté autres pays.<br>Le pays n'est pas une cause d'annulation et il est proche de la nationalité. |
| Décision du binôme, si l'alerte déclenche une relance | Oui. Pas de préjudice significatif pour les clients. |
| Décision du binôme, si l'alerte déclenche un acompte | Acceptable uniquement sous forme d'acompte remboursable, demandé lors de la relance.<br>Le client qui vient récupère son acompte : le préjudice est limité. |

## Tâche 4 — Atténuer

`entrainer(train, colonnes)` réentraîne la même recette que la v2.1 (seuil 0,2) sur les colonnes données.

In [11]:
# 1. Réentraîner sans `country`
colonnes_sans_pays = [colonne for colonne in INPUT_FEATURES if colonne != "country"]
modele_sans_pays = entrainer(train, colonnes_sans_pays)

# 2. Mesurer par groupe (Portugal / Autres pays)
mesures_groupes_v21 = mesures_par_groupe(v21, test, INPUT_FEATURES, groupe)
mesures_groupes_sans_pays = mesures_par_groupe(modele_sans_pays, test, colonnes_sans_pays, groupe)

comparaison_par_groupe = pd.concat({
    "v2.1": mesures_groupes_v21,
    "sans country": mesures_groupes_sans_pays,
})

# 3. Comparer la performance globale
# Chaque réservation reçoit la même étiquette "Ensemble" : mesures_par_groupe ne voit
# qu'un seul groupe et renvoie une seule ligne, calculée sur tout le jeu de test.
tout_le_test = pd.Series("Ensemble", index=test.index)

mesures_globales_v21 = mesures_par_groupe(v21, test, INPUT_FEATURES, tout_le_test)
mesures_globales_sans_pays = mesures_par_groupe(modele_sans_pays, test, colonnes_sans_pays, tout_le_test)

comparaison_globale = pd.concat([
    mesures_globales_v21.rename(index={"Ensemble": "v2.1"}),
    mesures_globales_sans_pays.rename(index={"Ensemble": "sans country"}),
])

display(comparaison_par_groupe, comparaison_globale)

/home/user1/labs/M2-B2-biais-wassim-frederic/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


effectif  annulations_reelles  taux_alertes  rappel  \
v2.1         Autres pays   12871.0                0.258         0.223   0.530   
             Portugal       6823.0                0.620         0.744   0.931   
sans country Autres pays   12871.0                0.258         0.438   0.695   
             Portugal       6823.0                0.620         0.551   0.776   

                          fausses_alertes  precision  
v2.1         Autres pays            0.117      0.612  
             Portugal               0.438      0.776  
sans country Autres pays            0.348      0.409  
             Portugal               0.184      0.873

,effectif,annulations_reelles,taux_alertes,rappel,fausses_alertes,precision
v2.1,19694.0,0.383,0.404,0.755,0.185,0.717
sans country,19694.0,0.383,0.477,0.741,0.313,0.595


- Sans country, l'écart se resserre sur le taux d'alertes (55 % contre 44 %, au lieu de 74 % contre 22 %). Sur les fausses alertes, il change de sens : les Portugais fiables passent de 44 % à 18 %, les autres de 12 % à 35 %.
- Le modèle sans country détecte presque autant d'annulations (rappel 74 % contre 76 %), mais alerte plus souvent à tort : les fausses alertes passent de 19 % à 31 % et la précision de 72 % à 60 %.

### À investiguer

Peut-on deviner le pays sans la colonne `country` ? Choisir une méthode pour répondre, et dire quelles variables sont en cause.

In [13]:
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder

# 1. Nouvelle cible : le client est-il portugais ?
est_portugais_train = train["country"].eq("PRT")
est_portugais_test = test["country"].eq("PRT")

# 2. Classifieur qui devine le pays à partir des autres colonnes
encodage = make_column_transformer(
    (OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), make_column_selector(dtype_include=object)),
    remainder="passthrough",
)
detecteur_pays = make_pipeline(encodage, HistGradientBoostingClassifier(random_state=0))
detecteur_pays.fit(train[colonnes_sans_pays], est_portugais_train)

# 3. Qualité de la devinette : 0,5 = hasard, 1 = parfait
proba_portugais = detecteur_pays.predict_proba(test[colonnes_sans_pays])[:, 1]
print("AUC :", round(roc_auc_score(est_portugais_test, proba_portugais), 3))

# 4. Variables en cause : chute d'AUC quand on mélange chaque variable
importance = permutation_importance(
    detecteur_pays, test[colonnes_sans_pays], est_portugais_test,
    scoring="roc_auc", n_repeats=5, random_state=0,
)
chute_auc_moyenne = importance["importances_mean"]
variables_en_cause = pd.Series(chute_auc_moyenne, index=colonnes_sans_pays).sort_values(ascending=False)
variables_en_cause.round(3)

AUC : 0.834


deposit_type                   0.123
market_segment                 0.047
hotel                          0.041
lead_time                      0.023
stays_in_weekend_nights        0.016
stays_in_week_nights           0.013
customer_type                  0.011
adr                            0.010
previous_cancellations         0.006
total_of_special_requests      0.002
adults                         0.001
required_car_parking_spaces    0.000
children                      -0.000
arrival_date_month            -0.004
dtype: float64

### ✏️ Interprétation

1. L'écart a-t-il disparu, diminué, changé de sens ? Pour quelles mesures ?
 **L'écart n'a pas disparu.** Il diminue pour le taux d'alertes (52 points d'écart avant, 11 après) et pour le rappel (40 points avant, 8 après). Il change de sens pour les fausses alertes : 44 % contre 12 % avant, 18 % contre 35 % après. Il se creuse pour la précision (87 % contre 41 %)
2. Qui gagne et qui perd au retrait de `country` ?
 **Gagnants : les clients portugais qui viennent**, signalés à tort dans 18 % des cas au lieu de 44 %. **Perdants : les clients fiables des autres pays**, signalés à tort dans 35 % des cas au lieu de 12 %. Le service réservations perd aussi : ses alertes sur les autres pays sont justes 4 fois sur 10 au lieu de 6.
3. Quel est le coût en performance, et est-il acceptable ?
 **Le rappel est stable (74 % contre 76 %), mais la précision passe de 72 % à 60 % et les fausses alertes de 19 % à 31 %.** Ce coût est acceptable pour une simple relance.
4. Faut-il interroger le client sur une autre variable ? Laquelle ?
**Oui : `deposit_type`.** Dans le jeu de test, les 2 426 réservations « Non Refund » sont toutes portugaises, et c'est la variable qui permet le mieux de deviner le pays (AUC 0,83 sans `country`). 